# Gaussian processes and kernels

Family 7 is an independent mathematical module backed by scikit-learn. This notebook fits generated observations, compares latent and noisy predictive uncertainty, and draws **jointly** across query points.

Inputs are `(n,d)` features and a scalar response `(n,)`. There is no time interpretation or hidden train/test split. See [the contracts](../docs/gaussian_processes.md) for the full catalog and noise conventions.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "convexity").is_dir():
    root = next(p for p in root.parents if (p / "convexity").is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from convexity.math import gaussian_processes as gp

rng = np.random.default_rng(42)
x = np.linspace(-3, 3, 45)[:, None]
y = np.sin(1.5*x[:, 0]) + rng.normal(0, .12, len(x))
query = np.linspace(-3.5, 3.5, 100)[:, None]

## Fit a covariance model

A constant kernel scales the Matérn covariance. Its value is a variance multiplier. Length scale and amplitude are selected by training marginal likelihood; `nu` remains fixed. The known training noise variance is supplied through `alpha`.

`restarts=1` adds one seeded optimizer start. The fitted object owns its state and leaves the initial kernel unchanged.

In [ ]:
kernel = gp.Constant(1., bounds=(.05, 10)) * gp.Matern(1., nu=2.5, bounds=(.1, 5))
model = gp.GaussianProcessRegressor(kernel, alpha=.12**2, restarts=1, seed=7).fit(x, y)
print("Initial:", kernel)
print("Fitted: ", model.kernel)
print("Log marginal likelihood:", model.log_marginal_likelihood())
print(model.diagnostics)
assert model.diagnostics.converged

## Latent function and future observations

`alpha` affects conditioning only. A future noisy observation has the same mean with an extra variance on its diagonal. The intervals below are **pointwise**, not simultaneous bands. They condition on fitted hyperparameters.

In [ ]:
latent = model.posterior(query)
observed = model.posterior(query, noise_variance=.12**2)
assert_allclose(observed.mean, latent.mean)
assert_allclose(observed.covariance - latent.covariance, .12**2*np.eye(len(query)), atol=1e-12)
assert_allclose(model.predict_std(query), latent.std, atol=1e-10)

latent_interval = latent.interval(.95)
observed_interval = observed.interval(.95)
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(query[:, 0], *observed_interval.T, color="#b9cde0", label="Future observation: 95% pointwise")
ax.fill_between(query[:, 0], *latent_interval.T, color="#75a5c7", alpha=.8, label="Latent function: 95% pointwise")
ax.plot(query[:, 0], np.sin(1.5*query[:, 0]), "--", color="#314b30", label="Generating function")
ax.plot(query[:, 0], latent.mean, color="#173e67", label="Posterior mean")
ax.scatter(x[:, 0], y, s=16, color="#3f4448", label="Training observations")
ax.set(xlabel="Input coordinate", ylabel="Response", title="Exact Gaussian-process regression")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

## Draw correlated function values

Each draw contains one jointly sampled value at every query row. The full covariance determines how neighboring values move together. Duplicate query points without independent noise share the same random value.

In [ ]:
draws = latent.sample(4, rng=np.random.default_rng(7))
assert draws.shape == (4, len(query))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(query[:, 0], draws.T, linewidth=1, alpha=.8)
axes[0].plot(query[:, 0], latent.mean, color="black", linewidth=2, label="Mean")
axes[0].set(xlabel="Input coordinate", ylabel="Response", title="Joint posterior samples")
axes[0].legend()
image = axes[1].imshow(latent.covariance, origin="lower", cmap="RdBu_r")
axes[1].set(xlabel="Query row", ylabel="Query row", title="Posterior covariance")
fig.colorbar(image, ax=axes[1], label="Covariance")
fig.tight_layout()
plt.show()

duplicate = model.posterior([[.4], [.4]])
paired = duplicate.sample(50, rng=np.random.default_rng(12))
assert_allclose(paired[:, 0], paired[:, 1], atol=1e-7)

## Fit an unknown noise level

A `WhiteNoise` component makes noise variance a kernel hyperparameter. Its contribution is already included in the predictive covariance, so there is no need to add it again with `noise_variance`. Its cross covariance is zero even for identical coordinates.

In [ ]:
noisy_kernel = gp.Constant(1., bounds=(.05, 10)) * gp.RBF(1., bounds=(.1, 5))
noisy_kernel += gp.WhiteNoise(.02, bounds=(1e-4, .5))
noisy_model = gp.gaussian_process_regression(x, y, kernel=noisy_kernel)
print("Learned covariance:", noisy_model.kernel)
print("Free parameters:", dict(zip(noisy_model.kernel.parameter_names, np.exp(noisy_model.kernel.theta))))
value, gradient = noisy_model.log_marginal_likelihood(eval_gradient=True)
print("Log marginal likelihood:", value)
print("Gradient in log coordinates:", gradient)
assert np.all(noisy_model.predict_std(query) > 0)

## Scope and module checks

This first implementation is dense, exact, scalar-response regression. It includes anisotropic RBF/Matérn, rational-quadratic, one-dimensional periodic, constant, dot-product and white-noise kernels, plus composition and custom scikit-learn kernels. It does not infer a model of dependence between multiple response variables.

Routine verification is local to this family:

```bash
python -m pytest -q tests/math/gaussian_processes
python -m pyright --pythonpath "$(command -v python)" convexity/math/gaussian_processes
python examples/gaussian_processes.py
```

Backend: [scikit-learn Gaussian processes](https://scikit-learn.org/stable/modules/gaussian_process.html).